# Reproduce GSViT single-frame phase recognition on Cholec80

This notebook reproduces the **Cholec80 single-frame phase-recognition** result reported in

> Schmidgall, Kim, Jopling & Krieger, *"General surgery vision transformer: A video pre-trained foundation model for general surgery"*, arXiv:2403.05949.

The paper reports **Accuracy 86.3 ± 5.9**, Precision 82.5 ± 7.1, Recall 84.7 ± 5.3 for the
**single-frame** GSViT model (13.7M tuneable + 12.4M frozen foundation weights) on Cholec80.

### Important facts about the official repo

The public repo (`SamuelSchmidgall/GSViT`) ships **only the *pretraining* code**
(`pretrain_model.py`, `dataloader_surgical.py`, `load_gsvit.py`). It does **not** contain any
Cholec80 fine-tuning/evaluation script. We reconstruct the Cholec80 pipeline from the paper text:

* Backbone: **EfficientViT-M5** (`embed_dim=[192,288,384]`, `depth=[1,3,4]`), head stripped.
  Its output is a latent of shape `(B, 384, 4, 4)` = **6144** features.
* The video-prediction decoder is detached; an MLP classifier is appended to the 6144-dim latent.
* Classifier: 3 linear layers, hidden sizes **2048** and **512**, **ELU** + layer/batch norm,
  **10% dropout**, output **7** classes.
* Backbone weights are **frozen** ("12.4M non-tuneable foundation-model weights"); only the
  ~13.7M-parameter head is trained.
* Training: **LR 3e-4, exponential decay, batch 128**, photometric distortion + Gaussian blur.

### What this notebook downloads

Section 1 downloads both required files automatically (resumable), so no manual download is needed:

1. **`GSViT.pkl`** — the released pretrained model (~1.36 GB, SharePoint).
2. **`cholec80.zip`** — the Cholec80 dataset (~69.8 GiB, public S3): 80 videos plus
   `phase_annotations/`.

> The 86.3% number **cannot be reached without the pretrained `GSViT.pkl`** — the whole point
> is the video-pretrained features. Training only on ImageNet features (the fallback below) is a
> useful smoke test of the code path, not a reproduction.

## 0. Setup

Runs on Colab or a local machine. The first cell detects Colab, mounts Google Drive (for
persistent storage), and checks the GPU. The second cell clones the repo and installs the extra
packages. We pin `timm` because the repo's EfficientViT code imports
`timm.models.layers.SqueezeExcite`, whose location moved in newer timm.

In [ ]:
import os, sys, shutil

# Detect Colab, mount Drive, check GPU and disk.
try:
    from google.colab import drive as _gdrive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    _gdrive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
else:
    DRIVE = "."

print("Colab:", IN_COLAB, "| DRIVE:", DRIVE)
if IN_COLAB:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (Runtime -> Change runtime type -> GPU)"
    print("GPU:", gpu)
_total, _used, _free = shutil.disk_usage("/" if IN_COLAB else ".")
print("local disk free:", round(_free / 1e9, 1), "GB")

In [ ]:
import os, sys, subprocess

# Fallback in case the Setup cell above was skipped.
try:
    IN_COLAB
except NameError:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

GSVIT_DIR = os.path.abspath("GSViT")
if not os.path.isdir(GSVIT_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/SamuelSchmidgall/GSViT.git", GSVIT_DIR], check=True)

if GSVIT_DIR not in sys.path:
    sys.path.insert(0, GSVIT_DIR)

# Extra packages. Do NOT reinstall torch/torchvision on Colab (already CUDA-enabled).
pkgs = ["timm==0.9.16", "opencv-python-headless", "numpy", "pandas",
        "matplotlib", "scikit-learn", "tqdm", "pillow", "requests"]
if not IN_COLAB:
    pkgs = ["torch", "torchvision"] + pkgs
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("repo + deps ready")

## 1. Download data & weights

Both files are downloaded automatically. Downloads are **resumable** — re-run a cell to continue
where it stopped.

| File | Size | Source |
|---|---|---|
| `cholec80.zip` | ~69.8 GiB (75 GB) | public S3 — **no login required** |
| `GSViT.pkl` | ~1.36 GB | SharePoint |

**Storage reality.** Free Google Drive is 15 GB and free Colab disk is ~78 GB, so a 69.8 GiB
zip fits in neither alone on the free tier. This notebook stores the zip and model on **Drive**
(you need ~100 GB of paid Drive space) and unzips to the **local** Colab disk each session.

In [ ]:
import requests, os
from tqdm.auto import tqdm

# Self-contained Colab/Drive detection (works even if the Setup cell was skipped).
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE = "/content/drive/MyDrive" if os.path.isdir("/content/drive") else "."
print("Colab:", IN_COLAB, "| DRIVE:", DRIVE)

GSVIT_URL = "https://livejohnshopkins-my.sharepoint.com/personal/sschmi46_jh_edu/_layouts/15/download.aspx?share=ERwRJGVUHRtKqvnSabuyKPgBfM8Hhe_Ks0EPTXg97nZ1Qw"
CHOLEC80_URL = "https://s3.unistra.fr/camma_public/datasets/cholec80/cholec80.zip"

# Targets on Drive. The zip survives across sessions; unzip goes to local disk for speed.
ZIP_DEST    = os.path.join(DRIVE, "cholec80", "cholec80.zip")
MODEL_DEST  = os.path.join(DRIVE, "cholec80", "GSViT.pkl")
CHOLEC80_DIR = "/content/cholec80" if IN_COLAB else "cholec80"

# The S3 endpoint is public. HTTP Basic Auth with the CAMMA credentials returns HTTP 400 (tested),
# so we download WITHOUT auth. If your network requires them, set auth=(user, pass):
# CHOLEC80_AUTH = ("camma_cholec80", "cholec80_unistra")

def download_file(url, dest, auth=None):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    have = os.path.getsize(dest) if os.path.exists(dest) else 0
    headers = {"Range": "bytes=%d-" % have} if have else {}
    with requests.get(url, stream=True, headers=headers, auth=auth,
                      allow_redirects=True, timeout=60) as r:
        if r.status_code not in (200, 206):
            raise RuntimeError("download failed: HTTP %d" % r.status_code)
        total = have + int(r.headers.get("Content-Length", 0) or 0)
        with open(dest, "ab" if have else "wb") as f, tqdm(
                total=total, initial=have, unit="B", unit_scale=True,
                desc=os.path.basename(dest)) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk); bar.update(len(chunk))
    print("saved:", dest)

In [ ]:
# (1) Dataset — ~69.8 GiB, resumable. Re-run to resume.
download_file(CHOLEC80_URL, ZIP_DEST)

In [ ]:
# (2) Unzip to the local (fast) disk. Slow for 75 GB; watch the free-space printout above.
import zipfile, time
os.makedirs(CHOLEC80_DIR, exist_ok=True)
t0 = time.time()
with zipfile.ZipFile(ZIP_DEST) as z:
    z.extractall(CHOLEC80_DIR)
print("unzipped to", CHOLEC80_DIR, "in", round((time.time() - t0) / 60, 1), "min")

In [ ]:
# (3) Locate the folder that contains videos/ and phase_annotations/.
def find_data_root(base):
    for root, dirs, _files in os.walk(base):
        if "videos" in dirs and "phase_annotations" in dirs:
            return root
    return base

CHOLEC80_ROOT = find_data_root(CHOLEC80_DIR)
print("CHOLEC80_ROOT =", CHOLEC80_ROOT)

In [ ]:
# (4) Model — ~1.36 GiB, resumable.
download_file(GSVIT_URL, MODEL_DEST)

## 2. Build the model

Reconstruct the GSViT classification model:

* `GSViTBackbone` — EfficientViT-M5 with the classification head removed, output `(B, 384, 4, 4)`.
* `CholecHead` — the 3-layer MLP classifier from the paper.
* `GSViTClassifier` — backbone (optionally frozen) + head.

In [ ]:
import torch
import torch.nn as nn
from EfficientViT.classification.model.build import EfficientViT_M5

LATENT_DIM = 384 * 4 * 4   # EfficientViT-M5 @224px -> (384, 4, 4) -> 6144

class GSViTBackbone(nn.Module):
    # EfficientViT-M5 minus the classification head. Returns (B, 384, 4, 4).
    # We re-attach the submodules by their original names (patch_embed, blocks1..3) so the
    # pretrained 'evit.*' state-dict keys line up 1:1.
    def __init__(self):
        super().__init__()
        m = EfficientViT_M5(num_classes=1000, pretrained=False)
        self.patch_embed = m.patch_embed
        self.blocks1 = m.blocks1
        self.blocks2 = m.blocks2
        self.blocks3 = m.blocks3

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.blocks1(x)
        x = self.blocks2(x)
        x = self.blocks3(x)
        return x                                  # (B, 384, 4, 4)


class CholecHead(nn.Module):
    # MLP classifier from the paper: 6144 -> 2048 -> 512 -> 7, ELU + LayerNorm, 10% dropout.
    def __init__(self, in_dim=LATENT_DIM, num_classes=7, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, 2048),
            nn.ELU(),
            nn.LayerNorm(2048),
            nn.Dropout(dropout),
            nn.Linear(2048, 512),
            nn.ELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class GSViTClassifier(nn.Module):
    def __init__(self, num_classes=7, freeze_backbone=True):
        super().__init__()
        self.backbone = GSViTBackbone()
        self.head = CholecHead(LATENT_DIM, num_classes)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        feat = self.backbone(x).flatten(1)        # (B, 6144)
        return self.head(feat)


if __name__ == "__main__":
    model = GSViTClassifier(freeze_backbone=True)
    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"total params: {n_total/1e6:.2f}M  |  trainable: {n_train/1e6:.2f}M")
    # expect ~26M total / ~13.7M trainable
    x = torch.randn(2, 3, 224, 224)
    with torch.no_grad():
        model.backbone.eval()
        print("latent shape:", model.backbone(x).shape, "| logits:", model(x).shape)

## 3. Load the pretrained `GSViT.pkl` into the backbone

The pickle may be a raw state-dict, a `{'model': ...}` wrapper, a pickled `nn.Module`, or the
auto-encoder state dict (`evit.*` + `decoder.*`). The loader below handles all of these, keeps
only backbone keys, strips prefixes (`evit.`, `module.`, `gsvit.`…), and drops the
classification head / decoder. It prints a diagnostic so you can verify the match.

In [ ]:
import pickle
from collections import OrderedDict

def load_backbone_from_pkl(model, pkl_path, verbose=True):
    obj = None
    errs = []
    # 1) torch.load (the pretrain script torch.save's a state_dict)
    try:
        obj = torch.load(pkl_path, map_location="cpu", weights_only=False)
    except Exception as e:
        errs.append(f"torch.load: {e}")
    # 2) raw pickle fallback
    if obj is None:
        try:
            with open(pkl_path, "rb") as f:
                obj = pickle.load(f)
        except Exception as e:
            errs.append(f"pickle.load: {e}")
    if obj is None:
        raise RuntimeError("could not load GSViT.pkl: " + " | ".join(errs))

    # unwrap containers
    sd = obj
    if isinstance(sd, nn.Module):
        sd = sd.state_dict()
    if isinstance(sd, dict) and "model" in sd and isinstance(sd["model"], dict):
        sd = sd["model"]
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    if not isinstance(sd, (dict, OrderedDict)):
        raise TypeError(f"unrecognised payload type: {type(obj)}")

    # normalise keys: strip wrappers, drop head/decoder, keep backbone only
    prefixes = ["module.", "gsvit.evit.", "evitae.evit.", "gsvit.", "evitae.", "evit."]
    drop = ("head", "decoder", "head_dist", "fc", "conv1", "conv2", "conv3", "conv4", "conv5",
            "bn1", "bn2", "bn3", "bn4", "res1", "res2", "bn1d", "sigmoid", "tanh")
    cleaned = {}
    for k, v in sd.items():
        if not hasattr(v, "shape"):
            continue
        kk = k
        for pfx in prefixes:
            if kk.startswith(pfx):
                kk = kk[len(pfx):]
                break
        if any(kk == d or kk.startswith(d + ".") for d in drop):
            continue
        cleaned[kk] = v

    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    if verbose:
        n_back = sum(1 for n, _ in model.named_parameters())
        print(f"matched: {n_back - len(missing)} / {n_back} backbone params")
        if missing:
            print("missing (first 8):", [m for m in missing][:8])
        if unexpected:
            print("unexpected (first 8):", [u for u in unexpected][:8])
    return missing, unexpected

# model = GSViTClassifier(freeze_backbone=True)
# load_backbone_from_pkl(model.backbone, "GSViT.pkl")

### Fallback: ImageNet-pretrained backbone (smoke test only)

If you do **not** have `GSViT.pkl` yet, this builds the backbone with ImageNet weights so you can
verify the dataset + training loop end-to-end. **This will not reproduce 86.3%.**

In [ ]:
def backbone_from_imagenet():
    bb = GSViTBackbone()
    m = EfficientViT_M5(num_classes=1000, pretrained="efficientvit_m5")  # downloads ImageNet ckpt
    sd = m.state_dict()
    sd = {k: v for k, v in sd.items() if not k.startswith("head")}
    bb.load_state_dict(sd, strict=False)
    return bb

# model = GSViTClassifier(freeze_backbone=True)
# model.backbone = backbone_from_imagenet()
print("Use the commented lines if you only have ImageNet weights.")

## 4. Cholec80 dataset

### Expected layout

```
cholec80/
├── videos/                    # video01.mp4 .. video80.mp4 (25 fps)
└── phase_annotations/         # video01-phase.txt .. video80-phase.txt
```

`videoXX-phase.txt` is one label per annotated frame (two columns `frame, phase`, or
`frame<TAB>phase`; a possible header line is skipped). Phase ids are 0..6 (1..7 is auto-shifted).
The mapping between video frames and annotation rows is done **proportionally in time**, so it is
correct whether the annotations are at 25 fps or 1 fps.

We decode each video once at **1 fps** and cache JPEG frames to disk for fast training.

In [ ]:
import re
import numpy as np

PHASE_NAMES = ["Preparation", "CalotTriangleDissection", "ClippingCutting",
               "GallbladderDissection", "GallbladderPackaging", "CleaningCoagulation",
               "GallbladderRetraction"]

def parse_phase_annotations(anno_path):
    labels = []
    with open(anno_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = re.split(r"[,;\t ]+", line)
            if parts and parts[0].lower() in {"frame", "frames", "phase", "idx", "index", "id", "tool"}:
                continue
            val = None
            for p in reversed(parts):          # phase is the last numeric column
                try:
                    val = int(float(p)); break
                except ValueError:
                    continue
            if val is not None:
                labels.append(val)
    labels = np.asarray(labels, dtype=np.int64)
    if labels.size and labels.min() == 1 and labels.max() == 7:
        labels = labels - 1                     # 1..7 -> 0..6
    return labels

In [ ]:
import cv2

def extract_video(video_path, anno_path, out_dir, sample_fps=1.0):
    # Decode a video at sample_fps, save JPEG frames, return per-frame phase ids.
    os.makedirs(out_dir, exist_ok=True)
    labels = parse_phase_annotations(anno_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {video_path}")
    vfps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, int(round(vfps / sample_fps)))
    out_labels, idx, saved = [], 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            if labels.size:
                # proportional time mapping: robust to annotation fps
                anno_idx = int(round(idx * (labels.size - 1) / max(1, n_frames - 1)))
                lab = int(labels[anno_idx])
            else:
                lab = -1
            out_labels.append(lab)
            cv2.imwrite(os.path.join(out_dir, f"{saved:06d}.jpg"), frame)
            saved += 1
        idx += 1
    cap.release()
    return out_labels

In [ ]:
import json, os

# CHOLEC80_ROOT is auto-detected in section 1 (folder with videos/ + phase_annotations/).
# Set it manually only if you skipped section 1:  CHOLEC80_ROOT = "/content/cholec80"
CACHE_DIR = os.path.join("/content", "cholec80_1fps") if "COLAB_GPU" in os.environ else "cholec80_1fps"
TRAIN_IDS = [f"video{i:02d}" for i in range(1, 41)]   # video01..video40
TEST_IDS  = [f"video{i:02d}" for i in range(41, 81)]  # video41..video80

def build_cache(ids, video_root, anno_root, cache_dir, sample_fps=1.0):
    os.makedirs(cache_dir, exist_ok=True)
    labels_map = {}
    for vid in ids:
        vdir = os.path.join(cache_dir, vid)
        lab_path = os.path.join(cache_dir, f"{vid}_labels.npy")
        if os.path.exists(lab_path):
            labels_map[vid] = np.load(lab_path).tolist()
            continue
        video_path = os.path.join(video_root, vid + ".mp4")
        anno_path = os.path.join(anno_root, vid + "-phase.txt")
        labs = extract_video(video_path, anno_path, vdir, sample_fps)
        np.save(lab_path, np.asarray(labs))
        labels_map[vid] = labs
        print(f"{vid}: {len(labs)} frames")
    with open(os.path.join(cache_dir, "labels.json"), "w") as f:
        json.dump(labels_map, f)
    return labels_map

# Uncomment once you have the dataset:
# labels_map = build_cache(TRAIN_IDS + TEST_IDS,
#                          os.path.join(CHOLEC80_ROOT, "videos"),
#                          os.path.join(CHOLEC80_ROOT, "phase_annotations"),
#                          CACHE_DIR)
print("Uncomment build_cache(...) after downloading Cholec80.")

In [ ]:
from PIL import Image

class Cholec80Dataset(torch.utils.data.Dataset):
    def __init__(self, video_ids, cache_dir, labels_map=None, transform=None):
        self.samples = []
        for vid in video_ids:
            vdir = os.path.join(cache_dir, vid)
            if not os.path.isdir(vdir):
                continue
            labs = (labels_map or {}).get(vid, [])
            for fn in sorted(os.listdir(vdir)):
                if fn.lower().endswith((".jpg", ".png")):
                    i = int(os.path.splitext(fn)[0])
                    lab = labs[i] if i < len(labs) else -1
                    if 0 <= lab <= 6:
                        self.samples.append((os.path.join(vdir, fn), lab))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, lab = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, lab

## 5. Transforms (photometric distortion + Gaussian blur, then 224px + ImageNet norm)

In [ ]:
import torchvision.transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # photometric distortion
    T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 6. DataLoaders (batch 128)

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 4

def make_loaders(labels_map, cache_dir=CACHE_DIR):
    train_ds = Cholec80Dataset(TRAIN_IDS, cache_dir, labels_map, train_transform)
    test_ds  = Cholec80Dataset(TEST_IDS,  cache_dir, labels_map, eval_transform)
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                           num_workers=NUM_WORKERS, pin_memory=True)
    test_dl  = torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                                           num_workers=NUM_WORKERS, pin_memory=True)
    print(f"train frames: {len(train_ds)} | test frames: {len(test_ds)}")
    return train_dl, test_dl

## 7. Training

Per the paper: Adam-family optimizer at **LR 3e-4 with exponential decay**, batch 128, cross-entropy.
The backbone is frozen; only the head trains. Optimizer and epoch count are **not** stated in the
paper — we use AdamW and a sensible epoch count, both editable below.

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 30
LR0 = 3e-4
GAMMA = 0.98            # per-epoch exponential decay factor: lr = LR0 * GAMMA**epoch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def train_one_epoch(model, dl, opt, crit, device):
    model.train()
    tot = cor = 0
    loss_sum = 0.0
    for x, y in tqdm(dl, desc="train", leave=False):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()
        loss_sum += loss.item() * x.size(0)
        cor += (logits.argmax(1) == y).sum().item()
        tot += x.size(0)
    return loss_sum / tot, cor / tot

In [ ]:
@torch.no_grad()
def evaluate(model, dl, device):
    model.eval()
    ys, ps = [], []
    for x, y in tqdm(dl, desc="eval", leave=False):
        x = x.to(device)
        ps.append(model(x).argmax(1).cpu())
        ys.append(y)
    return torch.cat(ys), torch.cat(ps)

In [ ]:
# --- main training entry point (uncomment after loaders + model are ready) ---
# from torch.optim import AdamW
# model = GSViTClassifier(freeze_backbone=True).to(DEVICE)
# load_backbone_from_pkl(model.backbone, "GSViT.pkl")
# train_dl, test_dl = make_loaders(labels_map)
# opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR0, weight_decay=1e-4)
# crit = nn.CrossEntropyLoss()
# for ep in range(EPOCHS):
#     lr = LR0 * (GAMMA ** ep)
#     for g in opt.param_groups:
#         g["lr"] = lr
#     tl, ta = train_one_epoch(model, train_dl, opt, crit, DEVICE)
#     print(f"epoch {ep:02d} lr {lr:.2e} | train loss {tl:.4f} acc {ta:.4f}")
# torch.save(model.state_dict(), "gsvit_cholec80.pt")
print("Uncomment the training block once model + loaders are ready.")

## 8. Evaluate and compare to the paper

Report **single-frame accuracy, macro precision, macro recall** and a confusion matrix, then compare
with the paper's `86.3 ± 5.9 / 82.5 / 84.7`. Note the `± 5.9` is a standard deviation (the paper
does not state whether it is over folds/seeds), so a single 40/40 run yields one point estimate;
see the k-fold note below.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

def report(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    cr = classification_report(y_true, y_pred, target_names=PHASE_NAMES, digits=3)
    print("=== Single-frame results ===")
    print(f"Accuracy : {acc*100:.2f}%")
    print(cr)
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(7)); ax.set_yticks(range(7))
    ax.set_xticklabels(PHASE_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(PHASE_NAMES, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()
    return acc

# y_true, y_pred = evaluate(model, test_dl, DEVICE)
# report(y_true.numpy(), y_pred.numpy())

### Reproducing the `± 5.9` (k-fold cross-validation)

Cholec80's official split is 40 train / 40 test, but many works also report 4-/5-fold CV, which is
the likely source of the paper's `± 5.9`. To reproduce the *mean ± std*, retrain the (fast, frozen
backbone) head over K folds of the **train** videos and average:

In [ ]:
def kfold_cv(ids, labels_map, k=5):
    import random
    random.Random(0).shuffle(ids)
    accs = []
    fold = len(ids) // k
    for f in range(k):
        val_ids = ids[f*fold:(f+1)*fold]
        tr_ids  = [v for v in ids if v not in val_ids]
        tr_ds = Cholec80Dataset(tr_ids,  CACHE_DIR, labels_map, train_transform)
        va_ds = Cholec80Dataset(val_ids, CACHE_DIR, labels_map, eval_transform)
        tr_dl = torch.utils.data.DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
        va_dl = torch.utils.data.DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        m = GSViTClassifier(freeze_backbone=True).to(DEVICE)
        load_backbone_from_pkl(m.backbone, "GSViT.pkl", verbose=False)
        opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()), lr=LR0, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        for ep in range(EPOCHS):
            for g in opt.param_groups:
                g["lr"] = LR0 * (GAMMA ** ep)
            train_one_epoch(m, tr_dl, opt, crit, DEVICE)
        yt, yp = evaluate(m, va_dl, DEVICE)
        accs.append(accuracy_score(yt.numpy(), yp.numpy()))
    accs = np.array(accs)
    print(f"CV accuracy: {accs.mean()*100:.2f} +- {accs.std()*100:.2f} %  (folds={accs*100})")
    return accs

# kfold_cv(TRAIN_IDS, labels_map, k=5)

## 9. Assumptions & caveats (things the paper/repo leave unspecified)

The following were **not** stated in the paper or repo and are our best-guess defaults; adjust if
you have better information:

| Choice | Our default | Notes |
|---|---|---|
| Optimizer | AdamW, wd 1e-4 | paper omits it; pretraining used Adam |
| Epochs | 30 | paper omits it |
| Exponential-decay rate | 0.98/epoch | "3e-4 with exponential decay", exact schedule unspecified |
| Input resolution | 224×224 | backbone native size; repo dataloader resizes to 224 |
| Normalization | ImageNet mean/std | backbone is ImageNet-initialised |
| Frame rate | 1 fps | standard for single-frame Cholec80; paper omits it |
| Split | 40 train / 40 test (video01–40 / 41–80) | official Cholec80 split |
| Backbone | frozen | from "12.4M non-tuneable foundation weights" |
| Norm in head | LayerNorm | paper says "layer normalization on hidden units" (also mentions batch norm) |
| Augmentation | ColorJitter + GaussianBlur | paper defers full list "to the code repository", which lacks it |

If a single run lands close to ~86% single-frame accuracy, the claim is confirmed for that split;
the full `± 5.9` band comes from cross-validation.